In [37]:
# CELL 1: Install Dependencies (UPDATED)
!pip install groq flask pyngrok nest-asyncio markdown -q
!pip install --upgrade groq -q

In [38]:
import os
from groq import Groq
from flask import Flask, request, jsonify, render_template_string
from pyngrok import ngrok
import nest_asyncio
from datetime import datetime
import threading # Import threading module
import socket # Import socket module to find free port
import markdown
from google.colab import userdata

# Apply nest_asyncio untuk Colab
nest_asyncio.apply()

# ============================================================================
# KONFIGURASI API KEY
# ============================================================================
# Masukkan Groq API Key Anda di sini
# Dapatkan API Key di: https://console.groq.com/keys
GROQ_API_KEY = userdata.get('GROQ_API_KEY')  # GANTI DENGAN API KEY ANDA

# Ngrok Authtoken (opsional, untuk tunnel yang lebih stabil)
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')  # Opsional

# Set API Key
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Inisialisasi Groq Client
client = Groq(api_key=GROQ_API_KEY)

# ============================================================================
# KONFIGURASI MODEL GROQ
# ============================================================================
# Daftar model yang tersedia dan aktif (2026)
AVAILABLE_MODELS = {
    "llama-3.3-70b-versatile": {
        "name": "Llama 3.3 70B (RECOMMENDED)",
        "description": "Model terbaru, paling powerful untuk edukasi",
        "max_tokens": 32768
    },
    "llama-3.1-8b-instant": {
        "name": "Llama 3.1 8B Instant",
        "description": "Cepat dan efisien untuk respons sederhana",
        "max_tokens": 8192
    },
    "llama-3.1-70b-versatile": {
        "name": "Llama 3.1 70B Versatile",
        "description": "High quality untuk tugas kompleks",
        "max_tokens": 32768
    },
    "mixtral-8x7b-32768": {
        "name": "Mixtral 8x7B",
        "description": "Long context, bagus untuk penjelasan panjang",
        "max_tokens": 32768
    }
}

# Model default yang digunakan
DEFAULT_MODEL = "llama-3.3-70b-versatile"

# ============================================================================
# KONFIGURASI EDUCATION BOT
# ============================================================================
SYSTEM_PROMPT = """
Anda adalah Education Bot, asisten AI yang ahli dalam bidang pendidikan dan pembelajaran.

KARAKTERISTIK:
- Gunakan bahasa yang formal namun ramah dan mudah dipahami
- Fokus pada domain pengetahuan pendidikan, pembelajaran, dan akademik
- Berikan penjelasan yang komprehensif dan terstruktur
- Sertakan contoh konkret untuk memudahkan pemahaman
- Dorong semangat belajar dan rasa ingin tahu pengguna
- Sesuaikan tingkat kesulitan penjelasan dengan pertanyaan pengguna

TOPIK YANG DIKUASAI:
- Matematika, Sains, Bahasa, Sejarah, dan mata pelajaran umum
- Strategi belajar dan teknik pembelajaran efektif
- Bimbingan akademik dan konsultasi pendidikan
- Penjelasan konsep dari dasar hingga advanced
- Tips dan trik mengerjakan soal
- Rekomendasi sumber belajar

GAYA KOMUNIKASI:
- Profesional dan edukatif
- Sabar dan suportif
- Memberikan motivasi belajar
- Menggunakan analogi untuk penjelasan yang kompleks

FORMAT RESPON:
- Hindari paragraf utuh yang panjang. Sajikan informasi dalam format yang terstruktur dan mudah dicerna.
- Gunakan daftar berpoin (bullet points), daftar bernomor, sub-judul, atau format outline kapan pun sesuai.
- Pisahkan bagian-bagian penting dengan jelas untuk kemudahan membaca.

Selalu berikan respons yang relevan, akurat, dan mendidik.
"""

# ============================================================================
# CHAT HISTORY MANAGEMENT
# ============================================================================
class EducationChatBot:
    def __init__(self, model_name=DEFAULT_MODEL):
        self.conversation_history = []
        self.system_message = {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
        self.conversation_history.append(self.system_message)
        self.model_name = model_name
        self.max_tokens = AVAILABLE_MODELS.get(model_name, {}).get("max_tokens", 32768)

    def add_to_history(self, role, content):
        """Menambahkan pesan ke history percakapan"""
        self.conversation_history.append({
            "role": role,
            "content": content
        })

    def get_response(self, user_message):
        """Mendapatkan respons dari Groq API"""
        # Tambahkan pesan user ke history
        self.add_to_history("user", user_message)

        try:
            # Panggil Groq API dengan model terbaru
            completion = client.chat.completions.create(
                model=self.model_name,
                messages=self.conversation_history,
                temperature=0.7,  # Kreativitas seimbang
                max_tokens=min(2048, self.max_tokens),  # Limit untuk respons yang baik
                top_p=1.0,
                stream=False
            )

            # Ambil respons assistant
            assistant_response = completion.choices[0].message.content

            # Tambahkan respons ke history
            self.add_to_history("assistant", assistant_response)

            return assistant_response

        except Exception as e:
            error_msg = f"Maaf, terjadi kesalahan: {str(e)}"
            # Hapus pesan user terakhir jika error
            if len(self.conversation_history) > 1:
                self.conversation_history.pop()
            return error_msg

    def clear_history(self):
        """Membersihkan history percakapan"""
        self.conversation_history = [self.system_message]

    def change_model(self, model_name):
        """Mengubah model yang digunakan"""
        if model_name in AVAILABLE_MODELS:
            self.model_name = model_name
            self.max_tokens = AVAILABLE_MODELS[model_name]["max_tokens"]
            return True
        return False

# Inisialisasi bot
education_bot = EducationChatBot()

# ============================================================================
# FLASK WEB APPLICATION
# ============================================================================
app = Flask(__name__)

# HTML Template untuk UI Chatbot
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>🎓 Education Bot - AI Learning Assistant</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }

        .container {
            max-width: 800px;
            margin: 0 auto;
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            overflow: hidden;
        }

        .header {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 30px;
            text-align: center;
        }

        .header h1 {
            font-size: 2em;
            margin-bottom: 10px;
        }

        .header p {
            opacity: 0.9;
            font-size: 0.95em;
        }

        .model-selector {
            background: rgba(255,255,255,0.1);
            padding: 10px 20px;
            border-radius: 10px;
            margin-top: 15px;
            display: inline-block;
        }

        .model-selector select {
            background: white;
            color: #333;
            border: none;
            padding: 8px 15px;
            border-radius: 5px;
            font-size: 0.9em;
            cursor: pointer;
        }

        .chat-container {
            height: 500px;
            overflow-y: auto;
            padding: 30px;
            background: #f8f9fa;
        }

        .message {
            margin-bottom: 20px;
            display: flex;
            align-items: flex-start;
        }

        .message.user {
            flex-direction: row-reverse;
        }

        .message-avatar {
            width: 40px;
            height: 40px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 1.5em;
            margin: 0 10px;
            flex-shrink: 0;
        }

        .message.bot .message-avatar {
            background: #667eea;
        }

        .message.user .message-avatar {
            background: #28a745;
        }

        .message-content {
            max-width: 70%;
            padding: 15px 20px;
            border-radius: 15px;
            line-height: 1.6;
        }

        .message.bot .message-content {
            background: white;
            border: 2px solid #e9ecef;
            border-top-left-radius: 5px;
        }

        .message.user .message-content {
            background: #667eea;
            color: white;
            border-top-right-radius: 5px;
        }

        .input-container {
            padding: 20px 30px;
            background: white;
            border-top: 2px solid #e9ecef;
            display: flex;
            gap: 10px;
        }

        #userInput {
            flex: 1;
            padding: 15px 20px;
            border: 2px solid #e9ecef;
            border-radius: 50px;
            font-size: 1em;
            outline: none;
            transition: border-color 0.3s;
        }

        #userInput:focus {
            border-color: #667eea;
        }

        button {
            padding: 15px 30px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 50px;
            cursor: pointer;
            font-size: 1em;
            font-weight: 600;
            transition: transform 0.2s, box-shadow 0.2s;
        }

        button:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 20px rgba(102, 126, 234, 0.4);
        }

        button:active {
            transform: translateY(0);
        }

        .typing {
            display: none;
            padding: 15px 20px;
            background: white;
            border-radius: 15px;
            border: 2px solid #e9ecef;
            margin-bottom: 20px;
        }

        .typing span {
            height: 8px;
            width: 8px;
            background: #667eea;
            border-radius: 50%;
            display: inline-block;
            margin: 0 2px;
            animation: typing 1.4s infinite;
        }

        .typing span:nth-child(2) { animation-delay: 0.2s; }
        .typing span:nth-child(3) { animation-delay: 0.4s; }

        @keyframes typing {
            0%, 60%, 100% { transform: translateY(0); }
            30% { transform: translateY(-10px); }
        }

        .timestamp {
            font-size: 0.75em;
            color: #6c757d;
            margin-top: 5px;
            text-align: right;
        }

        .welcome-message {
            text-align: center;
            padding: 40px;
            color: #6c757d;
        }

        .welcome-message h2 {
            color: #667eea;
            margin-bottom: 15px;
        }

        .features {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
            margin-top: 20px;
            text-align: left;
        }

        .feature-item {
            background: white;
            padding: 15px;
            border-radius: 10px;
            border-left: 4px solid #667eea;
        }

        /* Tambahkan CSS untuk format markdown */
    .message-content h1,
    .message-content h2,
    .message-content h3,
    .message-content h4 {
        margin: 10px 0;
        color: #667eea;
        font-weight: 600;
    }

    .message-content h1 { font-size: 1.5em; }
    .message-content h2 { font-size: 1.3em; }
    .message-content h3 { font-size: 1.1em; }

    .message-content ul,
    .message-content ol {
        margin: 10px 0;
        padding-left: 25px;
    }

    .message-content li {
        margin: 5px 0;
        line-height: 1.6;
    }

    .message-content p {
        margin: 10px 0;
        line-height: 1.8;
    }

    .message-content strong {
        color: #764ba2;
        font-weight: 600;
    }

    .message-content code {
        background: #f4f4f4;
        padding: 2px 6px;
        border-radius: 3px;
        font-family: 'Courier New', monospace;
        font-size: 0.9em;
    }

    .message-content pre {
        background: #2d2d2d;
        color: #f8f8f2;
        padding: 15px;
        border-radius: 8px;
        overflow-x: auto;
        margin: 10px 0;
    }

    .message-content pre code {
        background: none;
        padding: 0;
        color: inherit;
    }

    .message-content blockquote {
        border-left: 4px solid #667eea;
        margin: 10px 0;
        padding-left: 15px;
        color: #6c757d;
        font-style: italic;
    }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🎓 Education Bot</h1>
            <p>AI Learning Assistant - Teman Belajar Pintar Anda || by : Frencis Matheos Sarimole</p>
            <div class="model-selector">
                <label for="modelSelect" style="color: white; margin-right: 10px;">Model AI:</label>
                <select id="modelSelect" onchange="changeModel()">
                    <option value="llama-3.3-70b-versatile">Llama 3.3 70B (Recommended)</option>
                    <option value="llama-3.1-8b-instant">Llama 3.1 8B Instant</n>
                    <option value="llama-3.1-70b-versatile">Llama 3.1 70B Versatile</n>
                    <option value="mixtral-8x7b-32768">Mixtral 8x7B</n>
                </select>
            </div>
        </div>

        <div class="chat-container" id="chatContainer">
            <div class="welcome-message">
                <h2>Selamat Datang di Education Bot!</h2>
                <p>Saya siap membantu Anda dalam belajar. Tanyakan apapun tentang:</p>
                <div class="features">
                    <div class="feature-item">📚 Mata Pelajaran (Matematika, Sains, Bahasa, dll)</div>
                    <div class="feature-item">🎯 Strategi Belajar Efektif</div>
                    <div class="feature-item">💡 Penjelasan Konsep Sulit</div>
                    <div class="feature-item">📝 Tips Mengerjakan Soal</div>
                </div>
                <p style="margin-top: 20px;">Silakan mulai percakapan di bawah ini!</p>
            </div>
        </div>

        <div class="typing" id="typing">
            <span></span><span></span><span></span>
        </div>

        <div class="input-container">
            <input type="text" id="userInput" placeholder="Ketik pertanyaan Anda di sini..."
                   onkeypress="if(event.key === 'Enter') sendMessage()">
            <button onclick="sendMessage()">Kirim</button>
            <button onclick="clearChat()" style="background: #6c757d;">Reset</button>
        </div>
    </div>

    <script>
       // Di dalam function appendMessage, ubah bagian ini:
function appendMessage(role, content) {
    const chatContainer = document.getElementById('chatContainer');
    const messageDiv = document.createElement('div');
    messageDiv.className = `message ${role}`;

    const avatar = role === 'bot' ? '🤖' : '👨‍';
    const timestamp = new Date().toLocaleTimeString('id-ID', {
        hour: '2-digit',
        minute: '2-digit'
    });

    // Gunakan innerHTML untuk bot (karena sudah HTML), textContent untuk user
    const contentHTML = role === 'bot' ? content : content;

    messageDiv.innerHTML = `
        <div class="message-avatar">${avatar}</div>
        <div>
            <div class="message-content">${contentHTML}</div>
            <div class="timestamp">${timestamp}</div>
        </div>
    `;

    chatContainer.appendChild(messageDiv);
    chatContainer.scrollTop = chatContainer.scrollHeight;
}

        async function sendMessage() {
            const input = document.getElementById('userInput');
            const message = input.value.trim();

            if (!message) return;

            // Tampilkan pesan user
            appendMessage('user', message);
            input.value = '';

            // Tampilkan typing indicator
            document.getElementById('typing').style.display = 'block';
            document.getElementById('chatContainer').scrollTop =
                document.getElementById('chatContainer').scrollHeight;

            try {
                const response = await fetch('/chat', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ message: message })
                });

                const data = await response.json();

                // Sembunyikan typing indicator
                document.getElementById('typing').style.display = 'none';

                // Tampilkan respons bot
                appendMessage('bot', data.response);

            } catch (error) {
                document.getElementById('typing').style.display = 'none';
                appendMessage('bot', 'Maaf, terjadi kesalahan. Silakan coba lagi.');
                console.error('Error:', error);
            }
        }

        async function clearChat() {
            await fetch('/clear', { method: 'POST' });
            document.getElementById('chatContainer').innerHTML = `
                <div class="welcome-message">
                    <h2>Percakapan Dihapus</h2>
                    <p>Silakan mulai percakapan baru!</p>
                </div>
            `;
        }

        async function changeModel() {
            const modelSelect = document.getElementById('modelSelect');
            const modelName = modelSelect.value;

            try {
                const response = await fetch('/change-model', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ model: modelName })
                });

                const data = await response.json();
                if (data.success) {
                    appendMessage('bot', `✅ Model berhasil diubah ke: ${data.model_name}`);
                }
            } catch (error) {
                console.error('Error changing model:', error);
            }
        }
    </script>
</body>
</html>
"""

# ============================================================================
# FLASK ROUTES
# ============================================================================
@app.route('/')
def index():
    """Halaman utama chatbot"""
    return render_template_string(HTML_TEMPLATE)

# Update fungsi chat endpoint
@app.route('/chat', methods=['POST'])
def chat():
    """Endpoint untuk chat"""
    data = request.json
    user_message = data.get('message', '')

    if not user_message:
        return jsonify({'error': 'No message provided'}), 400

    # Dapatkan respons dari bot
    response = education_bot.get_response(user_message)

    # Convert markdown ke HTML
    html_response = markdown.markdown(response, extensions=['extra', 'codehilite'])

    return jsonify({
        'response': html_response,
        'timestamp': datetime.now().isoformat()
    })

@app.route('/clear', methods=['POST'])
def clear():
    """Endpoint untuk menghapus history chat"""
    education_bot.clear_history()
    return jsonify({'status': 'success'})

@app.route('/change-model', methods=['POST'])
def change_model():
    """Endpoint untuk mengubah model"""
    data = request.json
    model_name = data.get('model', DEFAULT_MODEL)

    success = education_bot.change_model(model_name)

    if success:
        return jsonify({
            'success': True,
            'model_name': AVAILABLE_MODELS[model_name]['name']
        })
    else:
        return jsonify({'success': False, 'error': 'Invalid model'}), 400

@app.route('/health')
def health():
    """Health check endpoint"""
    return jsonify({
        'status': 'running',
        'service': 'Education Bot',
        'model': education_bot.model_name
    })

# ============================================================================
# SETUP NGROK TUNNEL
# ============================================================================
def setup_ngrok(port):
    """Setup ngrok tunnel untuk akses publik"""

    # Set ngrok authtoken jika ada
    if NGROK_AUTHTOKEN != "34Nm8KveQKzpQbMdZIvfucuXxIO_jEkQV8UzpQhJZU6CCtvk":
        ngrok.set_auth_token(NGROK_AUTHTOKEN) # Corrected typo here

    # Kill any existing ngrok tunnels from pyngrok to avoid hitting limits
    ngrok.kill()

    # Buat tunnel ke port yang diberikan
    public_url = ngrok.connect(port)
    print(f"\n{'='*70}")
    print(f"🚀 EDUCATION BOT BERHASIL DIDEPLOY!")
    print(f"{'-'*70}")
    print(f"📱 Akses chatbot Anda di: {public_url}")
    print(f"🤖 Model yang digunakan: {education_bot.model_name}")
    print(f"{'='*70}")
    print(f"💡 Tips: Anda bisa mengubah model AI langsung dari UI chatbot")
    print(f"{'='*70}\n")

    return public_url

# Function to find an available port
def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('localhost', 0)) # Binds to port 0, letting OS choose a free port
        return s.getsockname()[1]

# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    # Find a free port
    port = find_free_port()

    # Setup ngrok with the chosen port
    public_url = setup_ngrok(port)

    # Jalankan Flask app in a separate thread
    print(f"🔄 Starting Flask server on port {port} in a separate thread...")
    thread = threading.Thread(target=app.run, kwargs={'host': '0.0.0.0', 'port': port, 'debug': False})
    thread.daemon = True
    thread.start()
    print("Flask server started. Access via ngrok tunnel.")


🚀 EDUCATION BOT BERHASIL DIDEPLOY!
----------------------------------------------------------------------
📱 Akses chatbot Anda di: NgrokTunnel: "https://preeffective-effectively-vanna.ngrok-free.dev" -> "http://localhost:35669"
🤖 Model yang digunakan: llama-3.3-70b-versatile
💡 Tips: Anda bisa mengubah model AI langsung dari UI chatbot

🔄 Starting Flask server on port 35669 in a separate thread...
Flask server started. Access via ngrok tunnel.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:35669
 * Running on http://172.28.0.12:35669
INFO:werkzeug:Press CTRL+C to quit
